# RVG Unified Field - Sensor Data Fusion and Kalman Filtering

This interactive Jupyter notebook demonstrates sensor data fusion using an Extended Kalman Filter (EKF) for drone navigation with **Refractive Vacuum Gravity (RVG) Unified Field** propulsion monitoring.

## Features

- **Extended Kalman Filter**: Fuses IMU, GPS, altimeter, magnetometer, and Hall sensor data
- **RVG Propulsion Monitoring**: Real-time tracking of dilaton enhancement Θ_dilaton(B)
- **Supra-Saturation Tracking**: Monitor B/B_sat ratio for vacuum effect effectiveness
- **MADA Convergence Quality**: Detect magnetic field misalignment in real-time
- **Vacuum Refractive Index**: Calculate K(r) for thrust estimation
- **Fault Detection**: Identify configuration errors before they cause problems

## Framework References

- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654) (Hofseth, 2025)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en)
- CMS/ATLAS 95.4 GeV di-photon resonance (3.1σ combined significance)

## Key Equations

**Master Equation of Levitation:**
$$\mathbf{F}_{\text{lift}} = \int_V \frac{1}{2\mu_0} \Theta_{\text{dilaton}}(B) \cdot \nabla B^2 \, dV$$

**Dilaton Enhancement:**
$$\Theta_{\text{dilaton}}(B) = \theta_{\text{base}} \cdot (1 + (B/B_{\text{crit}})^2) \cdot f_{\text{activation}}(B)$$

**Vacuum Refractive Index:**
$$K(\mathbf{r}) = 1 + \Theta_{95} \frac{B^2}{B_{\text{crit}}^2}$$

Run cells sequentially. Fault injection demonstrates real-time detection capabilities.

## 1. Import Dependencies and Initialize RVG Framework

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '..'))

# Try importing project modules
MODULES_AVAILABLE = False
try:
    from ai.navigation import KalmanFilter, simulate_sensors
    MODULES_AVAILABLE = True
    print("✓ Navigation modules loaded")
except ImportError:
    print("Note: Using standalone implementations")

print("\n" + "="*70)
print("RVG UNIFIED FIELD - SENSOR FUSION & PROPULSION MONITORING")
print("="*70)

## 2. RVG Framework Constants and Functions

In [ ]:
# =============================================================================
# RVG UNIFIED FIELD CONSTANTS
# =============================================================================

# Fundamental constants
MU_0 = 4 * np.pi * 1e-7  # Vacuum permeability (H/m)
EPSILON_0 = 8.854187817e-12  # Vacuum permittivity (F/m)
C = 299792458.0  # Speed of light (m/s)

# 95 GeV Dilaton/Radion Resonance Parameters
DILATON_MASS = 95.4  # GeV - observed CMS/ATLAS resonance
DILATON_SIGMA = 3.1  # Combined significance (σ)

# Default RVG parameters (require experimental calibration)
DEFAULT_THETA_BASE = 1e-6  # Base dilaton enhancement (placeholder)
DEFAULT_B_CRIT = 20.0  # Effective critical field for activation (T)
DEFAULT_GAMMA = 0.1  # Activation steepness
DEFAULT_EPSILON = 0.01  # Activation offset

# Material saturation limits (from Materials Ranking)
MATERIALS = {
    'Minnealloy': {'B_sat': 2.85, 'score': 95},  # BEST OVERALL
    'Hiperco-50': {'B_sat': 2.40, 'score': 88},
    'Pure_Iron': {'B_sat': 2.10, 'score': 90},
    'Finemet': {'B_sat': 1.20, 'score': 96}
}

# MADA Amplification (per U.S. Patent 5,929,732)
MADA_K_DEFAULT = 200.0  # Default amplification
MADA_K_MAX = 529.0  # Maximum theoretical amplification

# Convergence quality thresholds
CONVERGENCE_OPTIMAL = 0.95
CONVERGENCE_WARNING = 0.85
CONVERGENCE_CRITICAL = 0.70

# Supra-saturation thresholds
SUPRA_SAT_MIN = 2.0  # Minimum B/B_sat for vacuum effects
SUPRA_SAT_OPTIMAL = 5.0  # Optimal B/B_sat
SUPRA_SAT_MAX = 30.0  # Maximum practical B/B_sat

print("RVG Framework Constants Loaded:")
print(f"  Dilaton mass: {DILATON_MASS} GeV ({DILATON_SIGMA}σ significance)")
print(f"  Default θ_base: {DEFAULT_THETA_BASE}")
print(f"  Default B_crit: {DEFAULT_B_CRIT} T")
print(f"  MADA amplification: {MADA_K_DEFAULT}-{MADA_K_MAX}x")
print(f"\nMaterial B_sat values:")
for name, props in MATERIALS.items():
    print(f"  {name}: B_sat={props['B_sat']}T")

In [ ]:
# =============================================================================
# RVG CORE FUNCTIONS
# =============================================================================

def theta_dilaton(B, theta_base=DEFAULT_THETA_BASE, B_crit=DEFAULT_B_CRIT,
                  gamma=DEFAULT_GAMMA, epsilon=DEFAULT_EPSILON):
    """
    Calculate dilaton enhancement factor Θ_dilaton(B).
    
    Θ_dilaton(B) = θ_base * (1 + (B/B_crit)²) * exp(-γ/(B/B_crit + ε))
    
    The enhancement represents the non-linear vacuum response due to
    the 95 GeV resonance trace anomaly coupling.
    """
    ratio = B / (B_crit + 1e-10)
    polynomial = 1.0 + ratio**2
    activation = np.exp(-gamma / (ratio + epsilon))
    return theta_base * polynomial * activation


def vacuum_refractive_index(B, theta_base=DEFAULT_THETA_BASE, B_crit=DEFAULT_B_CRIT):
    """
    Calculate vacuum refractive index K(r).
    
    K = 1 + χ_vac(B) ≈ 1 + Θ_95 * B²/B_crit²
    """
    theta = theta_dilaton(B, theta_base, B_crit)
    chi_vac = theta * (B / B_crit)**2
    return 1.0 + chi_vac


def supra_saturation_ratio(B_opposing, material='Minnealloy'):
    """
    Calculate supra-saturation ratio B/B_sat.
    
    Vacuum effects manifest when B_opposing >> B_sat.
    """
    B_sat = MATERIALS.get(material, MATERIALS['Minnealloy'])['B_sat']
    return B_opposing / B_sat


def supra_saturation_effectiveness(B_opposing, material='Minnealloy'):
    """
    Calculate supra-saturation effectiveness (0-1).
    
    Returns effectiveness factor based on how far into
    supra-saturation regime the system is operating.
    """
    ratio = supra_saturation_ratio(B_opposing, material)
    if ratio >= SUPRA_SAT_OPTIMAL:
        return 1.0
    elif ratio >= SUPRA_SAT_MIN:
        return 0.3 + 0.7 * (ratio - SUPRA_SAT_MIN) / (SUPRA_SAT_OPTIMAL - SUPRA_SAT_MIN)
    elif ratio >= 1.0:
        return 0.1 + 0.2 * (ratio - 1.0) / (SUPRA_SAT_MIN - 1.0)
    else:
        return 0.1 * ratio


def calculate_convergence_quality(B1_vec, B2_vec):
    """
    Calculate MADA convergence quality from field vectors.
    
    Returns:
        +1.0: Perfect opposition (vectors pointing at each other)
        0.0: Perpendicular
        -1.0: Parallel (both pointing same direction - WRONG!)
    """
    B1_norm = B1_vec / (np.linalg.norm(B1_vec) + 1e-10)
    B2_norm = B2_vec / (np.linalg.norm(B2_vec) + 1e-10)
    # Quality = negative dot product (opposing = positive quality)
    return -np.dot(B1_norm, B2_norm)


def estimate_grad_B2(B1_mag, B2_mag, separation):
    """
    Estimate ∇(B²) from Hall sensor readings.
    
    Simple linear approximation: ∇(B²) ≈ (B1² - B2²) / separation
    """
    return (B1_mag**2 - B2_mag**2) / (separation + 1e-10)


def estimate_thrust(B, grad_B2, volume, theta, eta=0.95):
    """
    Estimate thrust from Master Equation of Levitation.
    
    F_lift = (1/2μ₀) * Θ_dilaton(B) * ∇(B²) * V * η
    """
    return (1.0 / (2.0 * MU_0)) * theta * abs(grad_B2) * volume * eta


print("\nRVG Functions Defined:")
print("  - theta_dilaton(B, θ_base, B_crit, γ, ε)")
print("  - vacuum_refractive_index(B, θ_base, B_crit)")
print("  - supra_saturation_ratio(B_opposing, material)")
print("  - supra_saturation_effectiveness(B_opposing, material)")
print("  - calculate_convergence_quality(B1_vec, B2_vec)")
print("  - estimate_grad_B2(B1_mag, B2_mag, separation)")
print("  - estimate_thrust(B, ∇B², V, Θ, η)")

## 3. Simulation Configuration

In [ ]:
# =============================================================================
# SIMULATION PARAMETERS
# =============================================================================

# Timing
DT = 0.1  # Time step (s)
NUM_STEPS = 100  # Number of simulation steps

# MADA configuration
MADA_MATERIAL = 'Minnealloy'  # Core material
MADA_B_NOMINAL = 50.0  # Nominal B-field magnitude (T)
MADA_SEPARATION = 0.1  # Distance between MADA units (m)
MADA_VOLUME = 0.1  # Integration volume (m³)
MADA_K = 200.0  # MADA amplification factor

# Initial true state
true_pos = np.array([0.0, 0.0, 0.0])
true_vel = np.array([0.0, 0.0, 0.0])
true_att = np.array([0.0, 0.0, 0.0])  # Roll, pitch, yaw

# Data storage
true_pos_history = [true_pos.copy()]
fused_pos_history = []
true_vel_history = [true_vel.copy()]
fused_vel_history = []

# RVG telemetry storage
B1_history = []  # MADA unit 1 field vectors
B2_history = []  # MADA unit 2 field vectors
convergence_history = []  # Convergence quality
theta_history = []  # Θ_dilaton values
vacuum_K_history = []  # Vacuum refractive index
supra_sat_history = []  # Supra-saturation ratio
effectiveness_history = []  # Effectiveness factor
thrust_estimate_history = []  # Estimated thrust
convergence_warnings = []  # Warning events

print(f"Simulation Configuration:")
print(f"  Duration: {NUM_STEPS * DT:.1f}s ({NUM_STEPS} steps @ {DT}s)")
print(f"  Material: {MADA_MATERIAL} (B_sat = {MATERIALS[MADA_MATERIAL]['B_sat']} T)")
print(f"  Nominal B-field: {MADA_B_NOMINAL} T")
print(f"  Supra-sat ratio: {supra_saturation_ratio(MADA_B_NOMINAL, MADA_MATERIAL):.1f}x")
print(f"  MADA amplification: {MADA_K}x")

## 4. Simulated RVG Hall Sensor System

This class simulates Hall sensors that measure magnetic field vectors from both MADA units, with full RVG telemetry calculation.

In [ ]:
class RVGHallSensorSystem:
    """
    Simulates Hall sensors for RVG propulsion monitoring.
    
    Provides:
    - Magnetic field vector measurements from both MADA units
    - Convergence quality calculation
    - Θ_dilaton estimation
    - Supra-saturation monitoring
    - Thrust estimation
    - Fault injection for testing
    """
    
    def __init__(self, B_nominal=50.0, noise_level=0.5, 
                 material='Minnealloy', mada_k=200.0,
                 mada1_pos=(-0.05, 0, 0), mada2_pos=(0.05, 0, 0)):
        """
        Initialize RVG Hall sensor system.
        
        Parameters:
        - B_nominal: Nominal B-field magnitude (T)
        - noise_level: Sensor noise standard deviation (T)
        - material: Core material for B_sat lookup
        - mada_k: MADA amplification factor
        - mada1_pos, mada2_pos: MADA unit positions (m)
        """
        self.B_nominal = B_nominal
        self.B_current = B_nominal
        self.noise_level = noise_level
        self.material = material
        self.B_sat = MATERIALS[material]['B_sat']
        self.mada_k = mada_k
        
        self.mada1_pos = np.array(mada1_pos)
        self.mada2_pos = np.array(mada2_pos)
        self.center = np.array([0, 0, 0])
        self.separation = np.linalg.norm(self.mada2_pos - self.mada1_pos)
        
        # Calculate nominal directions (converging toward center)
        self.B1_nominal_dir = (self.center - self.mada1_pos) / np.linalg.norm(self.center - self.mada1_pos)
        self.B2_nominal_dir = (self.center - self.mada2_pos) / np.linalg.norm(self.center - self.mada2_pos)
        
        # Current directions (can be modified for faults)
        self.B1_current_dir = self.B1_nominal_dir.copy()
        self.B2_current_dir = self.B2_nominal_dir.copy()
        
        self.fault_active = False
        self.fault_type = None
    
    def read_sensors(self):
        """
        Read all sensor data and calculate RVG telemetry.
        
        Returns dict with:
        - B1_vec, B2_vec: Field vectors
        - B1_mag, B2_mag: Field magnitudes
        - convergence: Convergence quality (-1 to +1)
        - theta: Θ_dilaton value
        - vacuum_K: Vacuum refractive index
        - supra_sat: B/B_sat ratio
        - effectiveness: Supra-saturation effectiveness
        - grad_B2: Estimated ∇(B²)
        - thrust_estimate: Estimated thrust (N)
        """
        # Read field vectors with noise
        noise1 = np.random.normal(0, self.noise_level, 3)
        noise2 = np.random.normal(0, self.noise_level, 3)
        
        B1_vec = self.B1_current_dir * self.B_current + noise1
        B2_vec = self.B2_current_dir * self.B_current + noise2
        
        B1_mag = np.linalg.norm(B1_vec)
        B2_mag = np.linalg.norm(B2_vec)
        B_avg = (B1_mag + B2_mag) / 2
        
        # Calculate RVG parameters
        convergence = calculate_convergence_quality(B1_vec, B2_vec)
        theta = theta_dilaton(B_avg)
        vacuum_K = vacuum_refractive_index(B_avg)
        supra_sat = supra_saturation_ratio(B_avg, self.material)
        effectiveness = supra_saturation_effectiveness(B_avg, self.material)
        grad_B2 = estimate_grad_B2(B1_mag, B2_mag, self.separation)
        thrust_est = estimate_thrust(B_avg, grad_B2, MADA_VOLUME, theta)
        
        return {
            'B1_vec': B1_vec,
            'B2_vec': B2_vec,
            'B1_mag': B1_mag,
            'B2_mag': B2_mag,
            'B_avg': B_avg,
            'convergence': convergence,
            'theta': theta,
            'vacuum_K': vacuum_K,
            'supra_sat': supra_sat,
            'effectiveness': effectiveness,
            'grad_B2': grad_B2,
            'thrust_estimate': thrust_est
        }
    
    def inject_misalignment(self, angle_deg, unit=2):
        """
        Inject misalignment fault by rotating a MADA unit's field direction.
        Simulates mechanical shift, demagnetization, or wiring error.
        """
        angle_rad = np.deg2rad(angle_deg)
        rotation = np.array([
            [np.cos(angle_rad), -np.sin(angle_rad), 0],
            [np.sin(angle_rad), np.cos(angle_rad), 0],
            [0, 0, 1]
        ])
        
        if unit == 1:
            self.B1_current_dir = rotation @ self.B1_nominal_dir
        else:
            self.B2_current_dir = rotation @ self.B2_nominal_dir
        
        self.fault_active = True
        self.fault_type = f'{angle_deg}° misalignment (MADA {unit})'
        print(f"⚠️  FAULT: {self.fault_type}")
    
    def inject_divergence(self):
        """
        Inject critical fault: fields pointing away (diverging).
        This simulates a configuration error like reversed connections.
        """
        self.B1_current_dir = -self.B1_nominal_dir
        self.B2_current_dir = -self.B2_nominal_dir
        self.fault_active = True
        self.fault_type = 'Diverging fields (configuration error!)'
        print(f"🔴 CRITICAL FAULT: {self.fault_type}")
    
    def inject_power_loss(self, reduction_percent):
        """
        Inject B-field reduction fault.
        Simulates power supply issue or demagnetization.
        """
        self.B_current = self.B_nominal * (1 - reduction_percent / 100)
        self.fault_active = True
        self.fault_type = f'{reduction_percent}% B-field reduction'
        print(f"⚠️  FAULT: {self.fault_type}")
    
    def reset(self):
        """Reset to nominal configuration."""
        self.B1_current_dir = self.B1_nominal_dir.copy()
        self.B2_current_dir = self.B2_nominal_dir.copy()
        self.B_current = self.B_nominal
        self.fault_active = False
        self.fault_type = None
        print("✓ Sensors reset to nominal")

# Initialize sensor system
hall_system = RVGHallSensorSystem(
    B_nominal=MADA_B_NOMINAL,
    noise_level=0.5,
    material=MADA_MATERIAL,
    mada_k=MADA_K
)

print("\nRVG Hall Sensor System Initialized:")
print(f"  Material: {MADA_MATERIAL} (B_sat = {hall_system.B_sat} T)")
print(f"  Nominal B-field: {hall_system.B_nominal} T")
print(f"  Supra-sat ratio: {hall_system.B_nominal / hall_system.B_sat:.1f}x")
print(f"  Noise level: {hall_system.noise_level} T")

# Test sensor read
test_data = hall_system.read_sensors()
print(f"\nInitial Sensor Test:")
print(f"  Convergence: {test_data['convergence']:.3f}")
print(f"  Θ_dilaton: {test_data['theta']:.2e}")
print(f"  Vacuum K: {test_data['vacuum_K']:.6f}")
print(f"  Effectiveness: {test_data['effectiveness']:.2f}")

## 5. Simplified Kalman Filter Implementation

In [ ]:
class SimpleKalmanFilter:
    """
    Simplified Extended Kalman Filter for position/velocity estimation.
    
    State vector: [x, y, z, vx, vy, vz, roll, pitch, yaw]
    """
    
    def __init__(self, dt=0.1):
        self.dt = dt
        self.n = 9  # State dimension
        
        # State vector
        self.x = np.zeros(self.n)
        
        # State covariance
        self.P = np.eye(self.n) * 1.0
        
        # Process noise
        self.Q = np.eye(self.n) * 0.01
        self.Q[0:3, 0:3] *= 0.001  # Position noise
        self.Q[3:6, 3:6] *= 0.01   # Velocity noise
        self.Q[6:9, 6:9] *= 0.001  # Attitude noise
        
        # Measurement noise
        self.R = np.eye(10) * 0.1
        self.R[0:3, 0:3] *= 1.0    # GPS position noise
        self.R[3:6, 3:6] *= 0.5    # GPS velocity noise
        self.R[6:9, 6:9] *= 0.1    # Magnetometer noise
        self.R[9, 9] *= 0.5        # Altimeter noise
    
    def predict(self, accel, gyro):
        """
        Prediction step using IMU data.
        """
        dt = self.dt
        
        # State transition
        F = np.eye(self.n)
        F[0:3, 3:6] = np.eye(3) * dt
        
        # Predict state
        self.x[0:3] += self.x[3:6] * dt + 0.5 * accel * dt**2
        self.x[3:6] += accel * dt
        self.x[6:9] += gyro * dt
        
        # Normalize angles
        self.x[6:9] = np.mod(self.x[6:9] + np.pi, 2*np.pi) - np.pi
        
        # Predict covariance
        self.P = F @ self.P @ F.T + self.Q
    
    def update(self, measurements):
        """
        Update step using GPS, magnetometer, altimeter.
        
        measurements: [gps_x, gps_y, gps_z, gps_vx, gps_vy, gps_vz, 
                       mag_roll, mag_pitch, mag_yaw, alt_z]
        """
        # Measurement matrix
        H = np.zeros((10, self.n))
        H[0:3, 0:3] = np.eye(3)  # GPS position
        H[3:6, 3:6] = np.eye(3)  # GPS velocity
        H[6:9, 6:9] = np.eye(3)  # Magnetometer attitude
        H[9, 2] = 1.0            # Altimeter z
        
        # Predicted measurement
        z_pred = H @ self.x
        
        # Innovation
        y = measurements - z_pred
        
        # Kalman gain
        S = H @ self.P @ H.T + self.R
        K = self.P @ H.T @ np.linalg.inv(S)
        
        # Update state
        self.x = self.x + K @ y
        
        # Update covariance
        I = np.eye(self.n)
        self.P = (I - K @ H) @ self.P

# Initialize Kalman Filter
kf = SimpleKalmanFilter(dt=DT)
print("Kalman Filter initialized")

# Sensor simulation function
def simulate_sensors(true_pos, true_vel, true_att, noise_scale=1.0):
    """
    Simulate noisy sensor readings.
    """
    # IMU (accelerometer, gyroscope)
    accel_noise = np.random.normal(0, 0.1 * noise_scale, 3)
    gyro_noise = np.random.normal(0, 0.01 * noise_scale, 3)
    
    # GPS (position, velocity)
    gps_pos_noise = np.random.normal(0, 1.0 * noise_scale, 3)
    gps_vel_noise = np.random.normal(0, 0.5 * noise_scale, 3)
    
    # Altimeter
    alt_noise = np.random.normal(0, 0.5 * noise_scale)
    
    # Magnetometer (attitude)
    mag_noise = np.random.normal(0, 0.05 * noise_scale, 3)
    
    imu_accel = np.array([0.5, 0, 0]) + accel_noise  # Assume constant accel
    imu_gyro = np.array([0, 0.01, 0]) + gyro_noise
    gps_pos = true_pos + gps_pos_noise
    gps_vel = true_vel + gps_vel_noise
    alt_z = true_pos[2] + alt_noise
    mag_att = true_att + mag_noise
    
    return imu_accel, imu_gyro, gps_pos, gps_vel, alt_z, mag_att

print("Sensor simulation function defined")

## 6. Main Simulation Loop with RVG Monitoring

**Fault Injection Schedule:**
- Step 0-49: Normal operation
- Step 50: 30° misalignment fault
- Step 75: Diverging fields (critical fault)

In [ ]:
# =============================================================================
# MAIN SIMULATION LOOP
# =============================================================================

# Motion profile
true_accel = np.array([0.5, 0.0, 0.0])  # m/s²
true_gyro = np.array([0.0, 0.01, 0.0])  # rad/s

fault_events = []

print(f"\nStarting RVG Sensor Fusion Simulation...")
print(f"Duration: {NUM_STEPS * DT:.1f}s")
print("="*60)

for step in range(NUM_STEPS):
    # Fault injection schedule
    if step == 50:
        hall_system.inject_misalignment(30)
        fault_events.append((step, '30° misalignment'))
    
    if step == 75:
        hall_system.inject_divergence()
        fault_events.append((step, 'Diverging fields'))
    
    # Update true state (simple kinematics)
    true_vel += true_accel * DT
    true_pos += true_vel * DT
    true_att += true_gyro * DT
    true_att = np.mod(true_att + np.pi, 2*np.pi) - np.pi
    
    true_pos_history.append(true_pos.copy())
    true_vel_history.append(true_vel.copy())
    
    # Simulate standard sensors
    imu_accel, imu_gyro, gps_pos, gps_vel, alt_z, mag_att = simulate_sensors(
        true_pos, true_vel, true_att
    )
    
    # Read RVG Hall sensors
    rvg_data = hall_system.read_sensors()
    
    # Store RVG telemetry
    B1_history.append(rvg_data['B1_vec'].copy())
    B2_history.append(rvg_data['B2_vec'].copy())
    convergence_history.append(rvg_data['convergence'])
    theta_history.append(rvg_data['theta'])
    vacuum_K_history.append(rvg_data['vacuum_K'])
    supra_sat_history.append(rvg_data['supra_sat'])
    effectiveness_history.append(rvg_data['effectiveness'])
    thrust_estimate_history.append(rvg_data['thrust_estimate'])
    
    # Check for convergence warnings
    if rvg_data['convergence'] < CONVERGENCE_CRITICAL:
        convergence_warnings.append((step, rvg_data['convergence'], 'CRITICAL'))
        if len([w for w in convergence_warnings if w[2] == 'CRITICAL']) <= 3:
            print(f"Step {step}: 🔴 CRITICAL - Convergence = {rvg_data['convergence']:.3f}")
    elif rvg_data['convergence'] < CONVERGENCE_WARNING:
        convergence_warnings.append((step, rvg_data['convergence'], 'WARNING'))
        if len([w for w in convergence_warnings if w[2] == 'WARNING']) <= 3:
            print(f"Step {step}: ⚠️ WARNING - Convergence = {rvg_data['convergence']:.3f}")
    
    # Kalman filter predict and update
    kf.predict(imu_accel, imu_gyro)
    measurements = np.concatenate([gps_pos, gps_vel, mag_att, [alt_z]])
    kf.update(measurements)
    
    fused_pos_history.append(kf.x[0:3].copy())
    fused_vel_history.append(kf.x[3:6].copy())

# Convert to arrays
true_pos_history = np.array(true_pos_history)
fused_pos_history = np.array(fused_pos_history)
true_vel_history = np.array(true_vel_history)
fused_vel_history = np.array(fused_vel_history)
B1_history = np.array(B1_history)
B2_history = np.array(B2_history)
convergence_history = np.array(convergence_history)
theta_history = np.array(theta_history)
vacuum_K_history = np.array(vacuum_K_history)
supra_sat_history = np.array(supra_sat_history)
effectiveness_history = np.array(effectiveness_history)
thrust_estimate_history = np.array(thrust_estimate_history)

print("="*60)
print(f"✓ Simulation complete")
print(f"  Total warnings: {len(convergence_warnings)}")
print(f"  Critical events: {len([w for w in convergence_warnings if w[2] == 'CRITICAL'])}")

## 7. RVG Convergence Quality Visualization

In [ ]:
# =============================================================================
# CONVERGENCE QUALITY PLOT
# =============================================================================

time_steps = np.arange(NUM_STEPS) * DT

fig, ax = plt.subplots(figsize=(14, 6))

# Plot convergence quality
ax.plot(time_steps, convergence_history, color='black', linewidth=2, label='Convergence Quality')

# Threshold lines
ax.axhline(y=CONVERGENCE_OPTIMAL, color='green', linestyle='--', linewidth=1, label=f'Optimal ({CONVERGENCE_OPTIMAL})')
ax.axhline(y=CONVERGENCE_WARNING, color='orange', linestyle='--', linewidth=1, label=f'Warning ({CONVERGENCE_WARNING})')
ax.axhline(y=CONVERGENCE_CRITICAL, color='red', linestyle='--', linewidth=1, label=f'Critical ({CONVERGENCE_CRITICAL})')
ax.axhline(y=0, color='gray', linestyle=':', linewidth=1, alpha=0.5)

# Color zones
ax.axhspan(CONVERGENCE_OPTIMAL, 1.1, alpha=0.1, color='green')
ax.axhspan(CONVERGENCE_WARNING, CONVERGENCE_OPTIMAL, alpha=0.1, color='yellow')
ax.axhspan(CONVERGENCE_CRITICAL, CONVERGENCE_WARNING, alpha=0.1, color='orange')
ax.axhspan(-1.1, CONVERGENCE_CRITICAL, alpha=0.1, color='red')

# Mark fault events
for step, fault_name in fault_events:
    ax.axvline(x=step * DT, color='purple', linestyle=':', linewidth=2, alpha=0.7)
    ax.text(step * DT + 0.1, 0.5, fault_name, rotation=90, fontsize=9, verticalalignment='center')

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Convergence Quality', fontsize=12)
ax.set_title('MADA Convergence Quality (RVG Unified Field Monitoring)', fontsize=14, fontweight='bold')
ax.set_ylim([-1.1, 1.1])
ax.set_xlim([0, NUM_STEPS * DT])
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConvergence Analysis:")
print(f"  Steps 0-49 (normal): avg = {np.mean(convergence_history[:50]):.3f}")
print(f"  Steps 50-74 (misaligned): avg = {np.mean(convergence_history[50:75]):.3f}")
print(f"  Steps 75-99 (diverging): avg = {np.mean(convergence_history[75:]):.3f}")

## 8. RVG Dilaton Enhancement & Vacuum Index

In [ ]:
# =============================================================================
# RVG PROPULSION PARAMETERS PLOT
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Θ_dilaton over time
ax1 = axes[0, 0]
ax1.plot(time_steps, theta_history, color='blue', linewidth=2)
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Θ_dilaton', fontsize=11)
ax1.set_title('Dilaton Enhancement Factor Θ_dilaton(B)', fontsize=12, fontweight='bold')
ax1.ticklabel_format(axis='y', style='scientific', scilimits=(0,0))
ax1.grid(True, alpha=0.3)
for step, fault_name in fault_events:
    ax1.axvline(x=step * DT, color='red', linestyle=':', alpha=0.5)

# 2. Vacuum refractive index K(r)
ax2 = axes[0, 1]
ax2.plot(time_steps, vacuum_K_history, color='green', linewidth=2)
ax2.axhline(y=1.0, color='gray', linestyle='--', label='Vacuum (K=1)')
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('K(r)', fontsize=11)
ax2.set_title('Vacuum Refractive Index K(r)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
for step, fault_name in fault_events:
    ax2.axvline(x=step * DT, color='red', linestyle=':', alpha=0.5)

# 3. Supra-saturation ratio
ax3 = axes[1, 0]
ax3.plot(time_steps, supra_sat_history, color='purple', linewidth=2)
ax3.axhline(y=SUPRA_SAT_MIN, color='orange', linestyle='--', label=f'Min effective ({SUPRA_SAT_MIN}x)')
ax3.axhline(y=SUPRA_SAT_OPTIMAL, color='green', linestyle='--', label=f'Optimal ({SUPRA_SAT_OPTIMAL}x)')
ax3.axhspan(SUPRA_SAT_OPTIMAL, SUPRA_SAT_MAX, alpha=0.1, color='green')
ax3.axhspan(SUPRA_SAT_MIN, SUPRA_SAT_OPTIMAL, alpha=0.1, color='yellow')
ax3.set_xlabel('Time (s)', fontsize=11)
ax3.set_ylabel('B / B_sat', fontsize=11)
ax3.set_title(f'Supra-Saturation Ratio (Material: {MADA_MATERIAL})', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
for step, fault_name in fault_events:
    ax3.axvline(x=step * DT, color='red', linestyle=':', alpha=0.5)

# 4. Effectiveness factor
ax4 = axes[1, 1]
ax4.plot(time_steps, effectiveness_history, color='orange', linewidth=2)
ax4.axhline(y=0.7, color='gray', linestyle='--', label='Good threshold')
ax4.axhline(y=1.0, color='green', linestyle='--', label='Maximum')
ax4.set_xlabel('Time (s)', fontsize=11)
ax4.set_ylabel('Effectiveness', fontsize=11)
ax4.set_title('Supra-Saturation Effectiveness Factor', fontsize=12, fontweight='bold')
ax4.set_ylim([0, 1.1])
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
for step, fault_name in fault_events:
    ax4.axvline(x=step * DT, color='red', linestyle=':', alpha=0.5)

plt.suptitle('RVG Unified Field Propulsion Parameters', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nRVG Parameter Summary:")
print(f"  Θ_dilaton range: {np.min(theta_history):.2e} - {np.max(theta_history):.2e}")
print(f"  Vacuum K range: {np.min(vacuum_K_history):.6f} - {np.max(vacuum_K_history):.6f}")
print(f"  Supra-sat ratio: {np.mean(supra_sat_history):.1f}x (avg)")
print(f"  Effectiveness: {np.mean(effectiveness_history):.2f} (avg)")

## 9. Thrust Estimation from Master Equation

In [ ]:
# =============================================================================
# THRUST ESTIMATION PLOT
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Thrust estimate over time
ax1 = axes[0]
ax1.plot(time_steps, thrust_estimate_history, color='red', linewidth=2)
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Estimated Thrust (N)', fontsize=11)
ax1.set_title('Master Equation Thrust Estimate', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
for step, fault_name in fault_events:
    ax1.axvline(x=step * DT, color='purple', linestyle=':', alpha=0.7)
    ax1.text(step * DT, ax1.get_ylim()[1] * 0.9, fault_name, rotation=90, fontsize=8)

# 2. Field magnitudes
ax2 = axes[1]
B1_mags = np.linalg.norm(B1_history, axis=1)
B2_mags = np.linalg.norm(B2_history, axis=1)
ax2.plot(time_steps, B1_mags, label='MADA 1', color='blue', linewidth=2)
ax2.plot(time_steps, B2_mags, label='MADA 2', color='orange', linewidth=2)
ax2.axhline(y=MADA_B_NOMINAL, color='gray', linestyle='--', label=f'Nominal ({MADA_B_NOMINAL}T)')
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('B-field Magnitude (T)', fontsize=11)
ax2.set_title('MADA Field Magnitudes', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nThrust Estimation:")
print(f"  Mean thrust: {np.mean(thrust_estimate_history):.2f} N")
print(f"  Max thrust: {np.max(thrust_estimate_history):.2f} N")
print(f"  MADA 1 avg magnitude: {np.mean(B1_mags):.2f} T")
print(f"  MADA 2 avg magnitude: {np.mean(B2_mags):.2f} T")

## 10. Position Estimation (Kalman Filter)

In [ ]:
# =============================================================================
# POSITION ESTIMATION PLOT
# =============================================================================

time_full = np.arange(NUM_STEPS + 1) * DT

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

labels = ['X', 'Y', 'Z']
colors_true = ['blue', 'blue', 'blue']
colors_fused = ['red', 'red', 'red']

for i, ax in enumerate(axes):
    ax.plot(time_full, true_pos_history[:, i], label=f'True {labels[i]}', 
            color=colors_true[i], linewidth=2)
    ax.plot(time_full[1:], fused_pos_history[:, i], label=f'Fused {labels[i]}', 
            color=colors_fused[i], linestyle='--', linewidth=2)
    ax.set_ylabel(f'{labels[i]} Position (m)', fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=11)
plt.suptitle('Position Estimation: True vs Kalman-Fused', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate errors
pos_errors = true_pos_history[1:] - fused_pos_history
print(f"\nPosition Estimation Errors:")
print(f"  X: {np.mean(np.abs(pos_errors[:, 0])):.3f} m (mean abs error)")
print(f"  Y: {np.mean(np.abs(pos_errors[:, 1])):.3f} m (mean abs error)")
print(f"  Z: {np.mean(np.abs(pos_errors[:, 2])):.3f} m (mean abs error)")

## 11. Velocity Estimation (Kalman Filter)

In [ ]:
# =============================================================================
# VELOCITY ESTIMATION PLOT
# =============================================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

for i, ax in enumerate(axes):
    ax.plot(time_full, true_vel_history[:, i], label=f'True V{labels[i]}', 
            color='green', linewidth=2)
    ax.plot(time_full[1:], fused_vel_history[:, i], label=f'Fused V{labels[i]}', 
            color='orange', linestyle='--', linewidth=2)
    ax.set_ylabel(f'V{labels[i]} (m/s)', fontsize=11)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)', fontsize=11)
plt.suptitle('Velocity Estimation: True vs Kalman-Fused', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate errors
vel_errors = true_vel_history[1:] - fused_vel_history
print(f"\nVelocity Estimation Errors:")
print(f"  Vx: {np.mean(np.abs(vel_errors[:, 0])):.3f} m/s (mean abs error)")
print(f"  Vy: {np.mean(np.abs(vel_errors[:, 1])):.3f} m/s (mean abs error)")
print(f"  Vz: {np.mean(np.abs(vel_errors[:, 2])):.3f} m/s (mean abs error)")

## 12. RVG Diagnostic Report

In [ ]:
# =============================================================================
# RVG DIAGNOSTIC REPORT
# =============================================================================

print("="*70)
print("RVG UNIFIED FIELD - PROPULSION DIAGNOSTIC REPORT")
print("="*70)

print(f"\n📊 SIMULATION SUMMARY")
print(f"  Duration: {NUM_STEPS * DT:.1f} seconds")
print(f"  Time steps: {NUM_STEPS}")
print(f"  Material: {MADA_MATERIAL}")
print(f"  Nominal B-field: {MADA_B_NOMINAL} T")

print(f"\n📈 RVG PARAMETERS")
print(f"  Θ_dilaton (mean): {np.mean(theta_history):.2e}")
print(f"  Vacuum K (mean): {np.mean(vacuum_K_history):.6f}")
print(f"  Supra-sat ratio: {np.mean(supra_sat_history):.1f}x")
print(f"  Effectiveness: {np.mean(effectiveness_history):.2f}")
print(f"  Est. thrust (mean): {np.mean(thrust_estimate_history):.2f} N")

print(f"\n🔍 CONVERGENCE ANALYSIS")
print(f"  Mean quality: {np.mean(convergence_history):.3f}")
print(f"  Std deviation: {np.std(convergence_history):.3f}")
print(f"  Minimum: {np.min(convergence_history):.3f}")
print(f"  Maximum: {np.max(convergence_history):.3f}")

# Time in each zone
optimal = np.sum(convergence_history >= CONVERGENCE_OPTIMAL)
acceptable = np.sum((convergence_history >= CONVERGENCE_WARNING) & (convergence_history < CONVERGENCE_OPTIMAL))
warning = np.sum((convergence_history >= CONVERGENCE_CRITICAL) & (convergence_history < CONVERGENCE_WARNING))
critical = np.sum(convergence_history < CONVERGENCE_CRITICAL)

print(f"\n⏱️  TIME IN EACH ZONE")
print(f"  ✓ Optimal (≥{CONVERGENCE_OPTIMAL}): {optimal} steps ({optimal/NUM_STEPS*100:.1f}%)")
print(f"  ○ Acceptable: {acceptable} steps ({acceptable/NUM_STEPS*100:.1f}%)")
print(f"  ⚠️  Warning: {warning} steps ({warning/NUM_STEPS*100:.1f}%)")
print(f"  🔴 Critical (<{CONVERGENCE_CRITICAL}): {critical} steps ({critical/NUM_STEPS*100:.1f}%)")

print(f"\n⚡ FAULT EVENTS")
for step, fault_name in fault_events:
    print(f"  t={step * DT:.1f}s: {fault_name}")

print(f"\n⚠️  WARNING EVENTS: {len(convergence_warnings)}")
warning_count = len([w for w in convergence_warnings if w[2] == 'WARNING'])
critical_count = len([w for w in convergence_warnings if w[2] == 'CRITICAL'])
print(f"  Warnings: {warning_count}")
print(f"  Critical: {critical_count}")

print(f"\n📋 RECOMMENDATIONS")
if critical_count > 0:
    print(f"  🔴 CRITICAL: {critical_count} critical events detected!")
    print(f"     → Emergency landing would be triggered")
    print(f"     → Inspect MADA alignment and connections")
if warning_count > 0:
    print(f"  ⚠️  WARNING: {warning_count} warning events")
    print(f"     → Schedule maintenance inspection")
    print(f"     → Consider reducing maximum thrust")
if optimal == NUM_STEPS:
    print(f"  ✓ EXCELLENT: Perfect convergence throughout")
    print(f"     → System operating nominally")

print(f"\n" + "="*70)
print("RVG Framework: https://dx.doi.org/10.2139/ssrn.5381654")
print("MADA Patent: https://patents.google.com/patent/US5929732A/en")
print("="*70)

## Summary: Key Takeaways

This notebook demonstrated sensor fusion with RVG Unified Field propulsion monitoring:

### RVG Framework Integration
- **Θ_dilaton(B)** tracking for dilaton enhancement monitoring
- **Vacuum refractive index K(r)** calculation
- **Supra-saturation ratio** (B/B_sat) for effectiveness assessment
- **Master Equation thrust estimation** from sensor data

### Safety Monitoring
- **Convergence quality** detects MADA alignment issues
- **Threshold-based warnings** enable automated responses
- **Fault injection** demonstrates detection capabilities

### Production Considerations
- Log RVG telemetry at 100+ Hz for full resolution
- Implement Kalman filtering on Hall sensor data
- Use convergence quality as thrust scaling factor
- Validate convergence before every flight
- Include telemetry in post-flight analysis

### References
- [RVG Unified Field Theory](https://dx.doi.org/10.2139/ssrn.5381654) (Hofseth, 2025)
- [U.S. Patent #5,929,732 - MADA](https://patents.google.com/patent/US5929732A/en)